### 説明変数生成



In [ ]:
from sklearn.preprocessing import StandardScaler
import glob

import os
import numpy as np
import sklearn.preprocessing

import matplotlib.pylab as plt
%matplotlib inline

g_rawfiles_mask = "Fe_random_calculated/*.cif"
g_descriptorfile = "../data_calculated/Fe_descriptor.csv"


In [ ]:
g_files =glob.glob(g_rawfiles_mask)
print(g_files)

**可視化**


説明変数変換の前に変換関数の可視化を行う。

In [ ]:
"""
plot G(r)
"""


def makesymfun2():
    """make type 2 symmetry function parameter

    Returns:
        dict: parameters of the symmetry functions
    """
    eps = 1e-5
    xmax = 6.0
    alist = [0.7]
    rplist = np.arange(2.4, 6.0, 0.6)
    rpmesh, amesh = make_mesh(rplist, alist)
    symfunparam = {}
    symfunparam["gauss2"] = {}
    symfunparam["gauss2"]["a"] = amesh
    symfunparam["gauss2"]["rp"] = rpmesh
    symfunparam["gauss2"]["list"] = {}
    symfunparam["gauss2"]["list"]["a"] = alist
    symfunparam["gauss2"]["list"]["rp"] = rplist

    labels = []
    label_str = ["a", "rp"]
    for a, rp in zip(amesh, rpmesh):
        # v = [a,rp]
        # value_str = list(map(str,v))
        # ss = [label_str[0],value_str[0],"_",label_str[1],value_str[1]]
        ss = "{}{:.2f}_{}{:.2f}".format(label_str[0], a, label_str[1], rp)
        labels.append("".join(ss))
    symfunparam["gauss2"]["labels"] = labels
    symfunparam["rc"] = xmax
    return symfunparam


def makesymfun():
    """make type 2 symmetry function parameter

    Returns:
        dict: symmetry function parameters
    """
    return makesymfun2()
    # I employed type 2


def make_mesh(alist, blist):
    """make 2D grid from 1D list alist, blist

    alist = [a1,a2]
    blist = [b1,b2,b3]
    ->
    ablist[:,0] = [a1,a2,a1,a2,a1,a2]
    ablist[:,1] = [b1,b1,b2,b2,b3,b3]
    Args:
        alist (np.array): list 1
        blist (np.array): list 2

    Returns:
        np.array: two dimensional mesh for the first axis
        np.array: two dimensional mesh for the second axis

    """
    ablist = np.meshgrid(alist, blist)
    ablist2 = []
    for a, b in zip(ablist[0].ravel(), ablist[1].ravel()):
        ablist2.append([a, b])
    ablist2 = np.array(ablist2)
    return ablist2[:, 0], ablist2[:, 1]

_DEBUG_ = False

def bsym_gauss2(xi, alist, rplist, xmax):
    """make 2-body gaussian terms

    Args:
        xi (np.array): a list of x
        alist (np.array): a list of parameter A
        rplist (np.array): a list of parameter rp
        xmax (float): r_max

    Returns:
        np.array: gaussian two body function at xi
    """
    xi = np.array(xi)
    x2 = (xi[:, None]-rplist[None, :])/alist[None, :]
    xi_ap = np.exp(-x2**2)*(np.cos(np.pi*xi[:, None]/xmax)+1.0)*0.5
    if _DEBUG_: 
        print("shape, xi,rplist,alist,xi_ap",xi.shape,rplist.shape,alist.shape, xi_ap.shape)
    return xi_ap


"""


4. visualiztion
"""


def plot_gauss2(symfunparam):
    """plot two body Gaussian terms

    Args:
        symfunparam(dict): a dict of symmetry function
    """
    amesh = symfunparam["gauss2"]["a"]
    rpmesh = symfunparam["gauss2"]["rp"]
    xmax = symfunparam["rc"]
    alist = symfunparam["gauss2"]["list"]["a"]
    rplist = symfunparam["gauss2"]["list"]["rp"]
    labels = symfunparam["gauss2"]["labels"]
    eps = 1e-5

    x = np.arange(0, xmax+eps, 0.1)
    y3 = bsym_gauss2(x, amesh, rpmesh, xmax)
    n = len(amesh)
    scaler = sklearn.preprocessing.MinMaxScaler()
    y4 = scaler.fit_transform(y3)
    fig, ax = plt.subplots()
    for i in range(n):
        ax.plot(x, y4[:, i], label=labels[i])
        ax.set_ylim((0, 1.0))
        ax.legend()
        ax.set_xlabel("R")
    fig.tight_layout()


g_symfunparam = makesymfun()
print("plot shape of symmetry functions")
plot_gauss2(g_symfunparam)
print("len=", len(g_symfunparam["gauss2"]["labels"]), ", param=", g_symfunparam)

**説明変数への変換**

原子構造を説明変数に変換する。

In [ ]:
# import pymatgen.io.xcrysden
from pymatgen.core import Structure
import copy
import pandas as pd
pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 60)




def filename2key(filename):
    """split filepath to base + extension

    Args:
        filename (str): filename
    Returns:
        str: filename without file extension
    """
    d, f = os.path.split(filename)
    b, ext = os.path.splitext(f)
    return b


def sitei(st, isite, rcut):
    """distance list and coordinate list with r_cut cuoff centered at isite

    Args:
        st (list): a list xsf structures
        isite (int): index of the site
        rcut (float): cutoff parameter

    Returns:
        list: a list of distance
        list: a list of their coordinates
    """
    distancelist = []
    coordlist = []
    for neigh in st.get_neighbors(st[isite], rcut):
        distancelist.append(neigh[1])
        coordlist.append(neigh[0].coords)
    return distancelist, coordlist


class make_sum_bsf:
    """
    summation of symmetry function
    """

    def __init__(self, dic):
        """initialization

        Args:
            dic (dict): parameters of the symmetry functions
        """

        self.amesh = dic["gauss2"]["a"]
        self.rpmesh = dic["gauss2"]["rp"]
        # rc
        self.xmax = dic["rc"]

    def transform(self, xi):
        """make symmetry function

        y_{jp} = sum_i exp(-((x_{ij}-rp)/a_p)**2 *(  cos(pi*x_{ij}/xmax) +1 )

        Args:
            xi (list): a list of x

        Returns:
            np.array: y_{jp}
        """

        xi = np.array(xi)
        xi_ap = bsym_gauss2(xi, self.amesh, self.rpmesh, self.xmax)

        sumi_xi_ap = np.sum(xi_ap, axis=0)

        return sumi_xi_ap


class make_rdf:
    def __init__(self, filename, rcut, normalize_length=False):
        """self.rdf, self.coords has atomic environments for each atom

        Args:
            filename (str): filename
            rcut (float): r cutoff
            normalize_length (bool, optional): True if normalize length. Defaults to False.

        Returns:
            [type]: [description]
        """
        self.filename = filename
        self.key = filename2key(filename)

        st = Structure.from_file(filename)
        self.st = st
        natom = len(st.sites)
        self.rdf = []
        self.coords = []
        for i in range(natom):
            a_rdf, a_coords = sitei(st, i, rcut)
            """
            return distance r_ij and corresponding atomic positions r_j centered at i site.
            I don't use r_j, which will be used to calculate three-body terms
            """
            self.rdf.append(a_rdf)
            self.coords.append(a_coords)
        """
        scale self.rdf with the nearest neighbour distance. 
        """
        if normalize_length:
            xmin = []
            for x in self.rdf:
                xmin.append(np.min(x))
            xmin = np.min(xmin)
            for i in range(natom):
                x = self.rdf[i]
                x = np.array(x)
                x /= xmin
                self.rdf[i] = list(x)

    def make_symfun3(self, symfunparam):
        """make symmetry functions centered at all the sites

        Args:
            symfunparam (dict): parameter of the symmetry function

        Returns:
            np.array: all the symmetry functions 
        """
        self.symfunparam = symfunparam

        sumrdf = make_sum_bsf(symfunparam)
        dlist = []
        for a_rdf in self.rdf:
            if _DEBUG_:
                print("a_rdf", len(a_rdf))
            y = sumrdf.transform(a_rdf)
            """
            covert them to descriptors
            """
            dlist.append(y)
        self.symfun2 = np.array(dlist)
        return self.symfun2


def load_structues(files, symfunparam, add_rdf=True):
    """load xsf files and make descriptors

    Args:
        files (list): a list of files
        symfunparam (dict): parameters of the symmetry funtion

    Returns:
        pd.DataFrame: descriptor, energy, property
        list: a list of rdf centered at all the sites
    """

    xmax = symfunparam["rc"]
    rdflist = []
    X = []
    R = []
    for filename in files:

        print(filename)
        rdf = make_rdf(filename, xmax) # rdf class
        rdflist.append(rdf.rdf)
        desc = rdf.make_symfun3(symfunparam)  # make explanatory variables
        
        key = rdf.key
        for i, (d,r) in enumerate(zip(desc,rdf.rdf)):
            x = [key, i] 
            x.extend(d)  # x = [key,site, explanatoryvariables_1, explanatoryvariables_2,...]
            if add_rdf:
                # The length of r may differ from each other.
                R.append(r) # x = [key,site, explanatoryvariables_1, explanatoryvariables_2,..., rdf as a list]
            X.append(x)
    
    X = np.array(X)
    index = ["key", "atom"]
    columns = copy.deepcopy(index)
    columns.extend(symfunparam["gauss2"]["labels"])
    df = pd.DataFrame(X, columns=columns)

    if add_rdf:
        df_R = pd.DataFrame({'rdf':R})
        df= pd.concat([df,df_R], axis=1)

    df2 = df.sort_values(by=index)
    df3 = df2.set_index(index, drop=True)

    return df3, rdflist, symfunparam["gauss2"]["labels"]


# 内部で説明変数へ変換する。
g_df, g_rdflist, g_descriptor_names = load_structues(g_files, g_symfunparam)

In [ ]:
g_df.index.levels[0]

In [ ]:
g_df

### histogram化したrdfの追加

In [ ]:
def df_hist(df,label="rdf"):
    """histogram化したrdfの追加を行う。

    Args:
        df (pd.DataFrame): データ。
        label (str, optional): カラム名. Defaults to "rdf".

    Returns:
        pd.DataFrame: hitogram化したrdfを追加したデータ。
        [str]: hitogramのカラム名。
        np.ndarray: histgram edges by np.histogram
    """
    rdf_min = []
    rdf_max = []
    for rdf in df["rdf"].values:
        rdf_min = np.min(rdf)
        rdf_max = np.max(rdf)
    rdf_min = np.min(rdf_min)
    rdf_max = np.max(rdf_max)
    eps = 1e-3
    rdf_lim = (rdf_min-eps, rdf_max+eps)
    rdf_lim
    bins = 20
    histlist = []
    for rdf in df["rdf"].values:
        hist, edges = np.histogram(rdf,range=rdf_lim, bins=bins)
        histlist.append(hist.tolist())
    Hist = np.array(histlist)
    Histcolumns = []
    for i in range(Hist.shape[1]):
        Histcolumns.append("hist{}".format(i+1))
    dfhist = pd.DataFrame(Hist,index=df.index, columns=Histcolumns)
    return dfhist, Histcolumns, edges
g_dfhist, g_hist_labels, g_hist_edges = df_hist(g_df)
g_dfhist

In [ ]:
# dfhistを結合する。

try:
    del g_df["rdf"]
except KeyError:
    pass
g_df = pd.concat([g_df,g_dfhist], axis=1)
g_df

可視化による"RDF" (実際は距離分布）の確認を行う。

In [ ]:
def plot_rdf(df, key, columns, hist_edges):
    """
    plot 'rdf'.
    
    Args:
        df (pd.DataFrame): データ。
        key (str): index name.
        columns (str): 説明変数名リスト.
        hist_edges (np.ndarray): histogram edges.
    """
    X = df.loc[key,columns]
    occur = np.sum(X, axis=0)
    n = occur.shape[0]
    center = (hist_edges[1:]+hist_edges[:-1])*0.5
    width = hist_edges[1]-hist_edges[0]
    
    fig, ax = plt.subplots(figsize=(3,1))
    
    ax.bar(center, occur, width=width)
    ax.set_title(key)
    ax. set_xlim((0, hist_edges[-1]))
    ax.set_xlabel("R index")
    ax.set_ylabel("occurrence")
    
print("histogram edge",g_hist_edges[0], g_hist_edges[-1])
for _key in ["Fe-bcc_01", "Fe-fcc_01", "Fe-hcp_01"]:
    plot_rdf(g_df,_key, g_hist_labels, g_hist_edges)

原子説明変数から結晶説明変数への変換

この場合は平均を取るだけです。

In [ ]:
g_df.columns

In [ ]:
def make_cell_df(df):
    """calculate cell averaged descriptor

    Args:
        df (pd.DataFrame): data with atomic descriptors

    Returns:
        pd.DataFrame: data with cell descriptors
    """

    v = []
    for id_ in df.index.levels[0]:
        x = df.loc[id_, :].astype(np.float64).values
        n = x.shape[0]
        x1 = np.sum(x, axis=0)/n
        v.append(x1)
    df2 = pd.DataFrame(v, index=df.index.levels[0], columns=df.columns)
    return df2

from copy import deepcopy
g_all_labels = deepcopy(g_descriptor_names)
g_all_labels.extend(g_hist_labels)
g_dfsample = make_cell_df(g_df[g_all_labels])
g_dfsample


ファイルに書き込むための調整を行う。

In [ ]:
g_df = g_dfsample.reset_index()
g_df

In [ ]:
def make_polytype_id(keys):
    """
    add polytype and its ids.
    
    Args:
        keys ([str]): keys.
    """
    name_list = []
    for key in keys:
        s = key.split("_")
        if len(s) == 1:
            name_list.append([s[0], -1])
        else:
            name_list.append([s[0], int(s[1])])
    df_meta = pd.DataFrame(name_list, columns=["polytype", "id"])
    return df_meta

g_df_meta = make_polytype_id(g_df["key"].values)
g_df_meta

In [ ]:
g_df2 = pd.concat([g_df,g_df_meta], axis=1)
g_df2

In [ ]:
g_df2["id"].values

ファイルへの書き込み

In [ ]:
import json
with open("condition.json","r") as f:
    g_nstructure = json.load(f)["nstructure"]


In [ ]:
def df_select_and_write(df2, nstructure):
    n = int(nstructure*0.8)
    print(n)
    df3 = df2.query("id<={}".format(n))
    df3.to_csv("../data_calculated/Fe2_descriptor.csv", index=False)
    print(df3.shape)
    df3 = df2.query("id>{}".format(n))
    df3.to_csv(
        "../data_calculated/Fe2_descriptor_newdata.csv", index=False)
    print(df3.shape)
df_select_and_write(g_df2,g_nstructure)

以下は動作確認のための読み込みを行う。

In [ ]:
g_dftmp = pd.read_csv("../data_calculated/Fe2_descriptor.csv", index_col=[0])
g_dftmp

In [ ]:
import seaborn as sns
sns.pairplot(g_dftmp[g_descriptor_names])

In [ ]:
g_dftmp = pd.read_csv("../data_calculated/Fe2_descriptor_newdata.csv")
g_dftmp

In [ ]:
g_dftmp.columns

In [ ]:
"done"